# Notebook 02 -- Data Preprocessing & Cleaning

**Project:** PropertyIQ -- Production-Grade Property Valuation System for Indian Banks

**Purpose:** Clean all 7 raw datasets discovered in Notebook 01, fix every known issue,
standardise to a common schema, merge into one master dataframe, and create the
train/val/drift split that powers drift detection in Notebook 05.

**Inputs:**
- 7 raw CSV files from `data/raw/`
- `outputs/inspection_report.json` (from Notebook 01)

**Known Issues to Fix:**
| Dataset   | Rows   | Known Issue                          |
|-----------|--------|--------------------------------------|
| Mumbai    | 76,038 | price_unit L/Cr -- must convert to Lakhs |
| V21       | 14,528 | City embedded in Location text       |
| Rental    |  4,746 | All dates Apr-Jul 2022 only          |
| Bengaluru | 13,320 | No date column                       |
| Pune      | 13,320 | No date column                       |
| Delhi     |  1,259 | Smaller dataset, careful outlier removal |
| RBI HPI   |     67 | Clean, ready                         |

**Outputs:**
- `data/processed/master_clean.csv` -- unified sale dataset
- `data/processed/train_baseline.csv`, `val.csv`, `drift_window.csv`
- `data/processed/rent_clean.csv`, `rent_train.csv`, `rent_drift.csv`
- `data/processed/hpi_macro.csv`
- `outputs/preprocessing_report.json`

## Cell 2 -- Imports & Constants

**Why:** Every dependency, path, and tuning parameter lives here so the entire
notebook can be reconfigured from one cell. Loading `inspection_report.json`
lets us confirm column names dynamically rather than hard-coding them.

In [ ]:
import json
import logging
import re
import sys
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats

# -- Logging ----------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("PropertyIQ.NB02")

# -- Paths ------------------------------------------------------------------
PROJECT_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f"Raw data directory not found: {DATA_DIR}"

# -- Load inspection report from NB01 --------------------------------------
INSPECTION_PATH = OUTPUT_DIR / "inspection_report.json"
try:
    with open(INSPECTION_PATH, "r", encoding="utf-8") as f:
        inspection = json.load(f)
    logger.info("Loaded inspection report from %s", INSPECTION_PATH)
except FileNotFoundError:
    logger.warning("inspection_report.json not found -- proceeding without it")
    inspection = {}

# -- Dataset filenames (CSVs on disk) ---------------------------------------
DATASETS = {
    "bengaluru": "Bengaluru_House_Data.csv",
    "pune":      "Pune house data.csv",
    "delhi":     "Delhi house data.csv",
    "mumbai":    "Mumbai House Prices.csv",
    "v21":       "Real Estate Data V21.csv",
    "rental":    "House_Rent_Dataset.csv",
    "hpi":       "QINR628BIS.csv",
}

# -- Sanity bounds ----------------------------------------------------------
PRICE_MIN_LAKH = 2.0
PRICE_MAX_LAKH = 100_000.0
SQFT_MIN = 100
SQFT_MAX = 50_000
BHK_MIN = 1
BHK_MAX = 10
RENT_MIN = 1_000
RENT_MAX = 500_000

# -- Standard output schema -------------------------------------------------
STANDARD_COLS = [
    "city", "locality", "property_type", "total_sqft", "bhk",
    "bath", "price_lakh", "price_sqft", "source", "data_period",
]

# -- Period mapping for RBI HPI enrichment ----------------------------------
PERIOD_HPI_MAP: Dict[str, float] = {}  # filled in Cell 10

# -- Conversion factors for sqft parsing ------------------------------------
SQFT_CONVERSIONS = {
    "sq. meter": 10.764,
    "sq meter":  10.764,
    "sq. yard":  9.0,
    "sq yard":   9.0,
    "grounds":   2400.0,
    "ground":    2400.0,
    "cents":     435.6,
    "cent":      435.6,
    "perch":     272.25,
    "guntha":    1089.0,
    "acres":     43560.0,
    "acre":      43560.0,
}

# Cities we can model
KNOWN_CITIES = [
    "Mumbai", "Delhi", "Bengaluru", "Bangalore", "Pune",
    "Hyderabad", "Chennai", "Kolkata", "Ahmedabad",
    "Noida", "Gurgaon", "Gurugram", "Thane", "Navi Mumbai",
]

logger.info("Constants initialised. PROCESSED_DIR: %s", PROCESSED_DIR)

## Cell 3 -- Core Utility Functions

**Why:** Every cleaning cell reuses these six functions. Centralising them here
means a bug fix propagates everywhere automatically. Each function is designed
around the *exact* data quirks found in Notebook 01's inspection.

In [ ]:
def standardize_city_name(raw_name: str) -> str:
    """Normalise city name variations to a canonical form.

    Uses an explicit lookup dict -- NOT fuzzy matching.
    Fast, predictable, no surprises.

    Args:
        raw_name (str): Raw city name from any dataset.

    Returns:
        str: Standardised city name.

    Example:
        >>> standardize_city_name("bangalore")
        'Bengaluru'
        >>> standardize_city_name("New Delhi")
        'Delhi'
    """
    CITY_MAP = {
        "bangalore":    "Bengaluru",
        "bengaluru":    "Bengaluru",
        "bengalore":    "Bengaluru",
        "mumbai":       "Mumbai",
        "bombay":       "Mumbai",
        "navi mumbai":  "Mumbai",
        "thane":        "Mumbai",
        "delhi":        "Delhi",
        "new delhi":    "Delhi",
        "noida":        "Delhi",
        "gurgaon":      "Delhi",
        "gurugram":     "Delhi",
        "pune":         "Pune",
        "hyderabad":    "Hyderabad",
        "chennai":      "Chennai",
        "madras":       "Chennai",
        "kolkata":      "Kolkata",
        "calcutta":     "Kolkata",
        "ahmedabad":    "Ahmedabad",
    }
    cleaned = str(raw_name).strip().lower()
    return CITY_MAP.get(cleaned, str(raw_name).strip().title())


def parse_sqft(sqft_raw) -> float:
    """Convert all sqft format variations to a float value in square feet.

    Handles plain numbers, range strings (midpoint), unit strings
    (Sq. Meter, Sq. Yards, Grounds, Cents, Perch, Acres), and NaN.

    Args:
        sqft_raw: Any sqft value from a dataset column.

    Returns:
        float: Area in sqft, or np.nan if unparseable.

    Example:
        >>> parse_sqft("1000 - 1500")
        1250.0
        >>> parse_sqft("111.48 Sq. Meter")
        1200.13
    """
    if pd.isna(sqft_raw):
        return np.nan

    val = str(sqft_raw).strip()

    # Check for unit conversions first
    val_lower = val.lower()
    for unit_key, factor in SQFT_CONVERSIONS.items():
        if unit_key in val_lower:
            num_part = re.sub(r"[^\d.]", " ", val_lower.split(unit_key)[0]).strip()
            try:
                return float(num_part) * factor
            except (ValueError, IndexError):
                return np.nan

    # Range string: "1000 - 1500"
    if "-" in val and not val.startswith("-"):
        parts = val.split("-")
        if len(parts) == 2:
            try:
                lo, hi = float(parts[0].strip()), float(parts[1].strip())
                return (lo + hi) / 2.0
            except ValueError:
                pass

    # Plain number
    try:
        return float(re.sub(r"[^\d.]", "", val))
    except (ValueError, TypeError):
        return np.nan


def parse_bhk(size_raw) -> float:
    """Extract BHK count from all format variations.

    Handles: "3 BHK", "3 Bedroom", "3BHK", "Studio", "1 RK", int, float.

    Args:
        size_raw: Raw BHK/size value from dataset.

    Returns:
        float: BHK count or np.nan if unparseable.

    Example:
        >>> parse_bhk("3 BHK")
        3.0
        >>> parse_bhk("Studio")
        1.0
    """
    if pd.isna(size_raw):
        return np.nan
    if isinstance(size_raw, (int, float)):
        return float(size_raw)

    val = str(size_raw).strip().lower()
    if "studio" in val or val == "rk":
        return 1.0
    match = re.search(r"(\d+)", val)
    return float(match.group(1)) if match else np.nan


def convert_price_to_lakh(price_val, unit: str) -> float:
    """Convert Mumbai price values to Lakhs using the price_unit column.

    CRITICAL: Mumbai uses 'L' for Lakhs and 'Cr' for Crores.
    Missing this conversion corrupts every downstream model.

    Args:
        price_val (float): Raw price number.
        unit (str): 'L' or 'Cr' from price_unit column.

    Returns:
        float: Price in Lakhs.

    Example:
        >>> convert_price_to_lakh(85.0, "L")
        85.0
        >>> convert_price_to_lakh(1.5, "Cr")
        150.0
    """
    if pd.isna(price_val) or pd.isna(unit):
        return np.nan
    unit_clean = str(unit).strip()
    if unit_clean == "Cr":
        return float(price_val) * 100.0
    elif unit_clean in ("L", "Lac", "Lakh"):
        return float(price_val)
    else:
        logger.warning("Unknown price unit '%s' -- treating as Lakhs", unit_clean)
        return float(price_val)


def extract_city_from_location(location: str, known_cities: List[str]) -> Optional[str]:
    """Extract a city name from a location string by searching for known cities.

    Used for the Real Estate V21 dataset where city is embedded in Location
    (e.g., 'Whitefield, Bangalore' or 'Bandra West, Mumbai').

    Args:
        location (str): Full location string.
        known_cities (List[str]): List of known city names to search for.

    Returns:
        Optional[str]: Matched + standardised city, or None if not found.

    Example:
        >>> extract_city_from_location("Bandra West, Mumbai", KNOWN_CITIES)
        'Mumbai'
    """
    if pd.isna(location):
        return None
    loc_lower = str(location).strip().lower()
    for city in known_cities:
        if city.lower() in loc_lower:
            return standardize_city_name(city)
    return None


def apply_period_split_by_price(df: pd.DataFrame, price_col: str,
                                 train_pct: float = 0.60) -> pd.DataFrame:
    """Create a data_period column using price distribution as a temporal proxy.

    Rationale: Post-COVID Indian RE prices are consistently higher than pre-COVID.
    Sorting by price_sqft and taking quantiles approximates the temporal split
    when no date column is available.

    Args:
        df (pd.DataFrame): Dataset with the price column.
        price_col (str): Column name to sort by.
        train_pct (float): Fraction for pre_covid bucket (default 0.60).

    Returns:
        pd.DataFrame: Same df with 'data_period' column added.

    Example:
        >>> df = apply_period_split_by_price(df_blr, 'price_sqft', 0.60)
    """
    transition_pct = 0.10
    q_low = df[price_col].quantile(train_pct)
    q_high = df[price_col].quantile(train_pct + transition_pct)

    conditions = [
        df[price_col] <= q_low,
        (df[price_col] > q_low) & (df[price_col] <= q_high),
        df[price_col] > q_high,
    ]
    choices = ["pre_covid", "transition", "post_covid"]
    df["data_period"] = np.select(conditions, choices, default="pre_covid")
    return df


def remove_outliers_iqr(df: pd.DataFrame, col: str,
                         multiplier: float = 2.5) -> pd.DataFrame:
    """Remove outliers using the IQR method.

    Uses multiplier=2.5 (tighter than standard 1.5) because Indian RE data
    has many data-entry errors with absurdly high or low values.

    Args:
        df (pd.DataFrame): Input dataframe.
        col (str): Column to check for outliers.
        multiplier (float): IQR multiplier (default 2.5).

    Returns:
        pd.DataFrame: Cleaned dataframe with outliers removed.

    Example:
        >>> df_clean = remove_outliers_iqr(df, 'price_sqft', 2.5)
    """
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr
    mask = (df[col] >= lower) & (df[col] <= upper)
    removed = (~mask).sum()
    if removed > 0:
        logger.info("IQR outlier removal on '%s': %d rows removed (bounds: %.1f -- %.1f)",
                     col, removed, lower, upper)
    return df[mask].copy()


def clean_city_dataset(df: pd.DataFrame, city: str, source: str,
                        col_map: Dict[str, str],
                        iqr_multiplier: float = 2.5) -> pd.DataFrame:
    """Standardised 11-step cleaning pipeline for a single-city sale dataset.

    Args:
        df (pd.DataFrame): Raw dataframe.
        city (str): Canonical city name.
        source (str): Source tag for provenance.
        col_map (Dict[str, str]): Mapping of raw column names to standard names.
            Must map to: locality, bhk_raw, total_sqft_raw, price_lakh, bath.
        iqr_multiplier (float): IQR multiplier for outlier removal (default 2.5).

    Returns:
        pd.DataFrame: Cleaned dataframe with STANDARD_COLS.

    Example:
        >>> df_clean = clean_city_dataset(df_blr, "Bengaluru", "bengaluru_kaggle",
        ...     {"location": "locality", "size": "bhk_raw", ...})
    """
    name = city.upper()
    raw_count = len(df)
    logger.info("%s: Starting cleaning -- %d raw rows", name, raw_count)

    # Step 1: Column mapping
    df_work = pd.DataFrame()
    df_work["locality"] = df[col_map["locality"]].astype(str).str.strip() if col_map.get("locality") else "Unknown"
    df_work["bhk_raw"] = df[col_map["bhk_raw"]] if col_map.get("bhk_raw") else np.nan
    df_work["total_sqft_raw"] = df[col_map["total_sqft_raw"]] if col_map.get("total_sqft_raw") else np.nan
    df_work["price_lakh"] = pd.to_numeric(df[col_map["price_lakh"]], errors="coerce")
    df_work["bath"] = pd.to_numeric(df[col_map.get("bath", "___missing___")] if col_map.get("bath") and col_map["bath"] in df.columns else pd.Series(np.nan, index=df.index), errors="coerce")

    # Property type detection
    if col_map.get("property_type") and col_map["property_type"] in df.columns:
        df_work["property_type"] = df[col_map["property_type"]].astype(str).str.strip()
    elif col_map.get("area_type") and col_map["area_type"] in df.columns:
        type_map = {"plot  area": "Plot", "plot area": "Plot"}
        df_work["property_type"] = df[col_map["area_type"]].astype(str).str.strip().str.lower().map(
            lambda x: type_map.get(x, "Apartment"))
    else:
        df_work["property_type"] = "Apartment"

    df_work["city"] = city
    df_work["source"] = source

    # Step 2: Parse sqft
    df_work["total_sqft"] = df_work["total_sqft_raw"].apply(parse_sqft)
    sqft_fails = df_work["total_sqft"].isna().sum()
    logger.info("%s: sqft parse -- %d failures", name, sqft_fails)

    # Step 3: Parse BHK
    df_work["bhk"] = df_work["bhk_raw"].apply(parse_bhk)
    bhk_fails = df_work["bhk"].isna().sum()
    logger.info("%s: BHK parse -- %d failures", name, bhk_fails)

    # Step 4: Calculate price_sqft
    df_work["price_sqft"] = np.where(
        df_work["total_sqft"] > 0,
        df_work["price_lakh"] * 100_000 / df_work["total_sqft"],
        np.nan,
    )

    # Step 5-7: Drop nulls in critical columns
    before = len(df_work)
    df_work = df_work.dropna(subset=["price_lakh"])
    null_price = before - len(df_work)

    before = len(df_work)
    df_work = df_work.dropna(subset=["total_sqft"])
    null_sqft = before - len(df_work)

    before = len(df_work)
    df_work = df_work.dropna(subset=["bhk"])
    null_bhk = before - len(df_work)

    after_nulls = len(df_work)
    logger.info("%s: Null drops -- price:%d, sqft:%d, bhk:%d", name, null_price, null_sqft, null_bhk)

    # Step 8: Sanity bounds
    before = len(df_work)
    df_work = df_work[
        (df_work["price_lakh"] >= PRICE_MIN_LAKH) &
        (df_work["price_lakh"] <= PRICE_MAX_LAKH) &
        (df_work["total_sqft"] >= SQFT_MIN) &
        (df_work["total_sqft"] <= SQFT_MAX) &
        (df_work["bhk"] >= BHK_MIN) &
        (df_work["bhk"] <= BHK_MAX)
    ].copy()
    bounds_dropped = before - len(df_work)
    after_bounds = len(df_work)
    logger.info("%s: Bounds filter -- %d rows dropped", name, bounds_dropped)

    # Recalculate price_sqft after filtering
    df_work["price_sqft"] = df_work["price_lakh"] * 100_000 / df_work["total_sqft"]

    # Step 9: IQR outlier removal on price_sqft
    before = len(df_work)
    df_work = remove_outliers_iqr(df_work, "price_sqft", iqr_multiplier)
    outlier_dropped = before - len(df_work)
    after_outliers = len(df_work)

    # Step 10: Period split
    df_work = apply_period_split_by_price(df_work, "price_sqft", 0.60)

    # Step 11: Select standard columns
    df_work["bhk"] = df_work["bhk"].astype(int)
    df_clean = df_work[STANDARD_COLS].copy()

    # -- Print summary --
    period_counts = df_clean["data_period"].value_counts()
    print(f"\n{'=' * 50}")
    print(f"  {name} CLEANING SUMMARY")
    print(f"{'=' * 50}")
    print(f"  Raw rows          : {raw_count:,}")
    print(f"  After null drops  : {after_nulls:,} (-{raw_count - after_nulls:,})")
    print(f"  After bounds      : {after_bounds:,} (-{after_nulls - after_bounds:,})")
    print(f"  After outliers    : {after_outliers:,} (-{outlier_dropped:,})")
    print(f"  Final clean rows  : {len(df_clean):,}")
    print(f"\n  data_period split:")
    for period in ["pre_covid", "transition", "post_covid"]:
        cnt = period_counts.get(period, 0)
        pct = cnt / len(df_clean) * 100 if len(df_clean) > 0 else 0
        print(f"    {period:<14}: {cnt:>6,} ({pct:>5.1f}%)")
    if len(df_clean) > 0:
        print(f"\n  price_sqft range  : Rs {df_clean['price_sqft'].min():,.0f} -- Rs {df_clean['price_sqft'].max():,.0f}")
        print(f"  avg price_sqft    : Rs {df_clean['price_sqft'].mean():,.0f}")
    print(f"{'=' * 50}\n")

    return df_clean


logger.info("Utility functions defined: 7 functions ready")

## Cell 4 -- Process Bengaluru Dataset

**Why:** Bengaluru is our primary single-city training dataset (13,320 rows).
Its `total_sqft` column can contain range strings like "1000-1500" and unit
strings like "111.48 Sq. Meter" -- both handled by `parse_sqft()`. The
`size` column uses "3 BHK" / "2 Bedroom" format parsed by `parse_bhk()`.

In [ ]:
try:
    df_blr_raw = pd.read_csv(DATA_DIR / DATASETS["bengaluru"])
    logger.info("Loaded Bengaluru: %d rows x %d cols", *df_blr_raw.shape)

    df_blr_clean = clean_city_dataset(
        df_blr_raw, city="Bengaluru", source="bengaluru_kaggle",
        col_map={
            "locality": "location",
            "bhk_raw": "size",
            "total_sqft_raw": "total_sqft",
            "price_lakh": "price",
            "bath": "bath",
            "area_type": "area_type",
        },
    )

    assert len(df_blr_clean) > 5000, f"Bengaluru: only {len(df_blr_clean)} rows after cleaning"
    assert df_blr_clean["price_lakh"].isna().sum() == 0, "Bengaluru: null prices remain"
    assert df_blr_clean["price_sqft"].isna().sum() == 0, "Bengaluru: null price_sqft remain"
    assert (df_blr_clean["city"] == "Bengaluru").all(), "Bengaluru: city mismatch"
    assert "pre_covid" in df_blr_clean["data_period"].values, "Bengaluru: missing pre_covid period"

    df_blr_clean.to_csv(PROCESSED_DIR / "bengaluru_clean.csv", index=False)
    logger.info("Bengaluru checkpoint saved: %d rows", len(df_blr_clean))

except Exception as exc:
    logger.error("Bengaluru processing failed: %s", exc)
    df_blr_clean = pd.DataFrame(columns=STANDARD_COLS)

## Cell 5 -- Process Pune Dataset

**Why:** Pune shares the same scraper schema as Bengaluru, but the location
column is named `site_location` instead of `location`. This single difference,
caught in Notebook 01's inspection, is handled here via the column mapping.

In [ ]:
try:
    df_pune_raw = pd.read_csv(DATA_DIR / DATASETS["pune"])
    logger.info("Loaded Pune: %d rows x %d cols", *df_pune_raw.shape)

    # Pune uses 'site_location' instead of 'location'
    loc_col = "site_location" if "site_location" in df_pune_raw.columns else "location"

    df_pune_clean = clean_city_dataset(
        df_pune_raw, city="Pune", source="pune_kaggle",
        col_map={
            "locality": loc_col,
            "bhk_raw": "size",
            "total_sqft_raw": "total_sqft",
            "price_lakh": "price",
            "bath": "bath",
            "area_type": "area_type",
        },
    )

    assert len(df_pune_clean) > 5000, f"Pune: only {len(df_pune_clean)} rows after cleaning"
    assert df_pune_clean["price_lakh"].isna().sum() == 0
    assert (df_pune_clean["city"] == "Pune").all()

    df_pune_clean.to_csv(PROCESSED_DIR / "pune_clean.csv", index=False)
    logger.info("Pune checkpoint saved: %d rows", len(df_pune_clean))

except Exception as exc:
    logger.error("Pune processing failed: %s", exc)
    df_pune_clean = pd.DataFrame(columns=STANDARD_COLS)

## Cell 6 -- Process Delhi Dataset

**Why:** Delhi has only 1,259 rows and a completely different schema (`Area`,
`BHK`, `Locality`, `Price`, `Per_Sqft`). We use a more lenient IQR multiplier
of 3.0 to avoid losing too much of this already-small dataset. The column
mapping adapts from Delhi's PascalCase naming to our lowercase standard.

In [ ]:
try:
    df_del_raw = pd.read_csv(DATA_DIR / DATASETS["delhi"])
    logger.info("Loaded Delhi: %d rows x %d cols", *df_del_raw.shape)

    # CRITICAL: Delhi Price column is in absolute rupees (e.g. 6,500,000)
    # Must convert to Lakhs (divide by 100,000) before cleaning pipeline
    df_del_raw["Price_Lakh"] = pd.to_numeric(df_del_raw["Price"], errors="coerce") / 100_000
    logger.info("Delhi: Converted prices from rupees to Lakhs (e.g. %s -> %.1f L)",
                df_del_raw["Price"].iloc[0], df_del_raw["Price_Lakh"].iloc[0])

    df_del_clean = clean_city_dataset(
        df_del_raw, city="Delhi", source="delhi_kaggle",
        col_map={
            "locality": "Locality",
            "bhk_raw": "BHK",
            "total_sqft_raw": "Area",
            "price_lakh": "Price_Lakh",
            "bath": "Bathroom",
            "property_type": "Type",
        },
        iqr_multiplier=3.0,  # more lenient -- small dataset
    )

    assert len(df_del_clean) > 400, f"Delhi: only {len(df_del_clean)} rows after cleaning"
    assert df_del_clean["price_lakh"].isna().sum() == 0
    assert (df_del_clean["city"] == "Delhi").all()

    df_del_clean.to_csv(PROCESSED_DIR / "delhi_clean.csv", index=False)
    logger.info("Delhi checkpoint saved: %d rows", len(df_del_clean))

except Exception as exc:
    logger.error("Delhi processing failed: %s", exc)
    df_del_clean = pd.DataFrame(columns=STANDARD_COLS)

## Cell 7 -- Process Mumbai Dataset (MOST COMPLEX)

**Why:** Mumbai is our largest dataset (76,038 rows) and has a critical
**price_unit** column: `L` = Lakhs (keep as-is), `Cr` = Crores (multiply by 100).
Missing this conversion would mean prices in the 1-10 range are treated as Lakhs
when they're actually Crores, corrupting every downstream model. We handle this
**first**, before any other processing step.

In [ ]:
mumbai_lac_rows = 0
mumbai_cr_rows = 0

try:
    df_mum_raw = pd.read_csv(DATA_DIR / DATASETS["mumbai"])
    logger.info("Loaded Mumbai: %d rows x %d cols", *df_mum_raw.shape)

    # STEP 0: Handle price_unit FIRST -- before anything else
    print(f"\n{'=' * 50}")
    print(f"  MUMBAI PRICE UNIT CONVERSION")
    print(f"{'=' * 50}")
    unit_dist = df_mum_raw["price_unit"].value_counts()
    for u, c in unit_dist.items():
        print(f"  {u:<6}: {c:>6,} rows ({c / len(df_mum_raw) * 100:.1f}%)")

    mumbai_lac_rows = int(unit_dist.get("L", 0))
    mumbai_cr_rows = int(unit_dist.get("Cr", 0))

    # Apply unit-aware conversion
    df_mum_raw["price_lakh_converted"] = df_mum_raw.apply(
        lambda row: convert_price_to_lakh(row["price"], row["price_unit"]), axis=1
    )

    print(f"\n  Mumbai price conversion:")
    print(f"    L  rows  : {mumbai_lac_rows:>6,} (price kept as-is)")
    print(f"    Cr rows  : {mumbai_cr_rows:>6,} (price x 100)")
    print(f"    Total    : {mumbai_lac_rows + mumbai_cr_rows:>6,} rows converted")
    print(f"{'=' * 50}\n")

    # Verify conversion: max should be in thousands, not ~150
    assert df_mum_raw["price_lakh_converted"].max() > 500, (
        "Mumbai Cr conversion likely failed -- max price_lakh is only "
        f"{df_mum_raw['price_lakh_converted'].max():.1f}"
    )
    logger.info("Mumbai price_unit conversion verified -- max: %.1f Lakhs",
                df_mum_raw["price_lakh_converted"].max())

    # Now run standard pipeline with the converted price column
    df_mum_raw["price"] = df_mum_raw["price_lakh_converted"]

    df_mum_clean = clean_city_dataset(
        df_mum_raw, city="Mumbai", source="mumbai_kaggle",
        col_map={
            "locality": "locality",
            "bhk_raw": "bhk",
            "total_sqft_raw": "area",
            "price_lakh": "price",
            "bath": "___no_bath___",  # Mumbai doesn't have bath column
            "property_type": "type",
        },
    )

    assert len(df_mum_clean) > 20000, f"Mumbai: only {len(df_mum_clean)} rows"
    assert df_mum_clean["price_lakh"].max() < PRICE_MAX_LAKH

    df_mum_clean.to_csv(PROCESSED_DIR / "mumbai_clean.csv", index=False)
    logger.info("Mumbai checkpoint saved: %d rows", len(df_mum_clean))

except Exception as exc:
    logger.error("Mumbai processing failed: %s", exc)
    df_mum_clean = pd.DataFrame(columns=STANDARD_COLS)

## Cell 8 -- Process Real Estate V21 Dataset

**Why:** V21 has 14,528 rows but **no explicit city column**. City information
is embedded in the `Location` text (e.g., "Avadi, Chennai"). We use
`extract_city_from_location()` to parse city names, then drop rows where
no known city is found. This dataset adds coverage for cities like Chennai
and Hyderabad that are missing from the other sale datasets.

In [ ]:
try:
    df_v21_raw = pd.read_csv(DATA_DIR / DATASETS["v21"])
    logger.info("Loaded V21: %d rows x %d cols", *df_v21_raw.shape)

    # Step 1: Extract city from Location
    df_v21_raw["city_extracted"] = df_v21_raw["Location"].apply(
        lambda x: extract_city_from_location(x, KNOWN_CITIES)
    )

    no_city = df_v21_raw["city_extracted"].isna().sum()
    has_city = len(df_v21_raw) - no_city
    logger.info("V21: City extraction -- %d found, %d unmatched (dropped)", has_city, no_city)

    df_v21_raw = df_v21_raw.dropna(subset=["city_extracted"]).copy()
    df_v21_raw["city_std"] = df_v21_raw["city_extracted"].apply(standardize_city_name)

    # Print city distribution
    print(f"\n  V21 cities found:")
    for city, cnt in df_v21_raw["city_std"].value_counts().items():
        print(f"    {city:<15}: {cnt:>5,}")

    # Handle V21 Price column -- may contain strings like "1.5 Cr" or "85 L"
    def parse_v21_price(price_raw) -> float:
        """Parse V21 price strings to Lakhs.

        Args:
            price_raw: Raw price value (may be string like '1.5 Cr' or '85 L').

        Returns:
            float: Price in Lakhs.

        Example:
            >>> parse_v21_price("1.5 Cr")
            150.0
        """
        if pd.isna(price_raw):
            return np.nan
        val = str(price_raw).strip()
        val_lower = val.lower()
        if "cr" in val_lower:
            num = re.sub(r"[^\d.]", "", val_lower.split("cr")[0])
            try:
                return float(num) * 100.0
            except ValueError:
                return np.nan
        elif "l" in val_lower or "lac" in val_lower or "lakh" in val_lower:
            num = re.sub(r"[^\d.]", "", val_lower.split("l")[0])
            try:
                return float(num)
            except ValueError:
                return np.nan
        else:
            try:
                return float(re.sub(r"[^\d.]", "", val))
            except (ValueError, TypeError):
                return np.nan

    df_v21_raw["price_lakh_parsed"] = df_v21_raw["Price"].apply(parse_v21_price)
    df_v21_raw["price_for_clean"] = df_v21_raw["price_lakh_parsed"]

    # Extract locality from Location (part before city)
    def extract_locality(location: str) -> str:
        """Extract locality part from V21 location string.

        Args:
            location (str): Full location string.

        Returns:
            str: Locality part (everything before the city name).

        Example:
            >>> extract_locality("Bandra West, Mumbai")
            'Bandra West'
        """
        if pd.isna(location):
            return "Unknown"
        parts = str(location).rsplit(",", 1)
        return parts[0].strip() if len(parts) > 1 else str(location).strip()

    df_v21_raw["locality_parsed"] = df_v21_raw["Location"].apply(extract_locality)

    # Build a temp df for clean_city_dataset -- need consistent columns
    df_v21_temp = pd.DataFrame({
        "locality": df_v21_raw["locality_parsed"],
        "bhk_raw": np.nan,  # V21 doesn't have BHK -- infer from area later
        "total_sqft": pd.to_numeric(df_v21_raw["Total_Area"], errors="coerce"),
        "price": df_v21_raw["price_for_clean"],
        "bath": pd.to_numeric(df_v21_raw["Baths"], errors="coerce"),
        "city_std": df_v21_raw["city_std"],
        "property_type": df_v21_raw.get("Property Title", "Apartment"),
    })

    # Infer BHK from area: rough heuristic
    # <600 sqft = 1BHK, 600-1000 = 2BHK, 1000-1500 = 3BHK, >1500 = 4BHK
    def infer_bhk_from_area(area: float) -> float:
        """Infer BHK from area using Indian RE heuristics.

        Args:
            area (float): Area in sqft.

        Returns:
            float: Estimated BHK count.

        Example:
            >>> infer_bhk_from_area(1200.0)
            3.0
        """
        if pd.isna(area) or area <= 0:
            return np.nan
        if area < 600:
            return 1.0
        elif area < 1000:
            return 2.0
        elif area < 1500:
            return 3.0
        elif area < 2500:
            return 4.0
        else:
            return 5.0

    df_v21_temp["bhk_raw"] = df_v21_temp["total_sqft"].apply(infer_bhk_from_area)

    # Process per-city with the standard pipeline
    v21_clean_parts = []
    for city_name in df_v21_temp["city_std"].unique():
        city_slice = df_v21_temp[df_v21_temp["city_std"] == city_name].copy()
        if len(city_slice) < 10:
            logger.info("V21: Skipping %s -- only %d rows", city_name, len(city_slice))
            continue
        # Build a pseudo-DataFrame for clean_city_dataset
        pseudo_df = pd.DataFrame({
            "locality": city_slice["locality"],
            "size": city_slice["bhk_raw"],
            "total_sqft": city_slice["total_sqft"],
            "price": city_slice["price"],
            "bath": city_slice["bath"],
            "property_type": city_slice["property_type"],
        })
        try:
            cleaned = clean_city_dataset(
                pseudo_df, city=city_name, source="v21_scrape",
                col_map={
                    "locality": "locality",
                    "bhk_raw": "size",
                    "total_sqft_raw": "total_sqft",
                    "price_lakh": "price",
                    "bath": "bath",
                    "property_type": "property_type",
                },
                iqr_multiplier=3.0,
            )
            v21_clean_parts.append(cleaned)
        except Exception as e:
            logger.warning("V21: %s cleaning failed: %s", city_name, e)

    if v21_clean_parts:
        df_v21_clean = pd.concat(v21_clean_parts, ignore_index=True)
    else:
        df_v21_clean = pd.DataFrame(columns=STANDARD_COLS)
        logger.warning("V21 dataset produced 0 usable rows -- skipping from master")

    df_v21_clean.to_csv(PROCESSED_DIR / "v21_clean.csv", index=False)
    logger.info("V21 checkpoint saved: %d rows", len(df_v21_clean))

except Exception as exc:
    logger.error("V21 processing failed: %s", exc)
    df_v21_clean = pd.DataFrame(columns=STANDARD_COLS)

## Cell 9 -- Process Rental Dataset

**Why:** The rental dataset is our only source with posting dates (Apr 13 -- Jul 11,
2022). However, the 89-day window falls **entirely** within the post-COVID period,
so we cannot use dates for a pre/post-COVID split. Instead, we use the `rent_sqft`
distribution as a proxy: bottom 50% represents the "pre_covid" baseline.

**LIMITATION (documented for transparency):** This proxy split is weaker than a
true temporal split and this must be noted in any model documentation.

In [ ]:
try:
    df_rent_raw = pd.read_csv(DATA_DIR / DATASETS["rental"])
    logger.info("Loaded Rental: %d rows x %d cols", *df_rent_raw.shape)
    raw_rent_count = len(df_rent_raw)

    # Step 1: Column mapping
    df_rent = pd.DataFrame()
    df_rent["bhk"] = pd.to_numeric(df_rent_raw["BHK"], errors="coerce")
    df_rent["total_sqft"] = pd.to_numeric(df_rent_raw["Size"], errors="coerce")
    df_rent["rent_monthly"] = pd.to_numeric(df_rent_raw["Rent"], errors="coerce")
    df_rent["locality"] = df_rent_raw["Area Locality"].astype(str).str.strip()
    df_rent["bath"] = pd.to_numeric(df_rent_raw["Bathroom"], errors="coerce")
    df_rent["city_raw"] = df_rent_raw["City"]
    df_rent["furnishing"] = df_rent_raw["Furnishing Status"]
    df_rent["source"] = "rent_kaggle"

    # Step 2: Date analysis -- transparent about limitation
    df_rent["listing_date"] = pd.to_datetime(df_rent_raw["Posted On"], errors="coerce")
    min_date = df_rent["listing_date"].min()
    max_date = df_rent["listing_date"].max()
    date_span = (max_date - min_date).days if pd.notna(min_date) and pd.notna(max_date) else 0

    print(f"\n{'=' * 50}")
    print(f"  RENTAL DATE RANGE ANALYSIS")
    print(f"{'=' * 50}")
    print(f"  Min date : {min_date.date() if pd.notna(min_date) else 'N/A'}")
    print(f"  Max date : {max_date.date() if pd.notna(max_date) else 'N/A'}")
    print(f"  Window   : {date_span} days (ALL post-COVID)")
    print(f"")
    print(f"  [!] LIMITATION: Cannot use dates for")
    print(f"  pre/post split. Using rent_sqft")
    print(f"  distribution proxy instead.")
    print(f"")
    print(f"  Bottom 50% rent_sqft --> 'pre_covid'")
    print(f"  Top 50%    rent_sqft --> 'post_covid'")
    print(f"{'=' * 50}\n")

    rental_date_limitation = (
        f"All rental dates within {date_span}-day window "
        f"({min_date.date()} to {max_date.date()}). "
        "Entirely post-COVID. Using rent_sqft distribution proxy for period split."
    )

    # Step 3: Fix city names
    df_rent["city"] = df_rent["city_raw"].apply(standardize_city_name)
    print("  Rental city mapping:")
    for raw_city in df_rent["city_raw"].unique():
        std_city = standardize_city_name(raw_city)
        cnt = (df_rent["city_raw"] == raw_city).sum()
        marker = " <-- MAPPED" if raw_city != std_city else ""
        print(f"    {raw_city:<15} --> {std_city:<15} ({cnt:,} rows){marker}")

    # Verify critical mapping
    assert "Bengaluru" in df_rent["city"].values or "Bangalore" not in df_rent["city"].values, \
        "City standardisation failed: 'Bangalore' should map to 'Bengaluru'"

    # Step 4: Calculate rent_sqft
    df_rent["rent_sqft"] = np.where(
        df_rent["total_sqft"] > 0,
        df_rent["rent_monthly"] / df_rent["total_sqft"],
        np.nan,
    )

    # Step 5: Sanity bounds
    before = len(df_rent)
    df_rent = df_rent.dropna(subset=["rent_monthly", "total_sqft", "bhk"])
    df_rent = df_rent[
        (df_rent["rent_monthly"] >= RENT_MIN) &
        (df_rent["rent_monthly"] <= RENT_MAX) &
        (df_rent["total_sqft"] >= SQFT_MIN) &
        (df_rent["total_sqft"] <= SQFT_MAX) &
        (df_rent["bhk"] >= BHK_MIN) &
        (df_rent["bhk"] <= BHK_MAX)
    ].copy()
    bounds_dropped = before - len(df_rent)
    logger.info("Rental: bounds filter -- %d rows dropped", bounds_dropped)

    # Recalculate after filtering
    df_rent["rent_sqft"] = df_rent["rent_monthly"] / df_rent["total_sqft"]

    # Step 6: Outlier removal
    before = len(df_rent)
    df_rent = remove_outliers_iqr(df_rent, "rent_sqft", 2.5)
    logger.info("Rental: outlier removal -- %d rows dropped", before - len(df_rent))

    # Step 7: Period split by rent_sqft distribution
    df_rent = apply_period_split_by_price(df_rent, "rent_sqft", 0.50)

    # Step 8: Final summary
    df_rent["bhk"] = df_rent["bhk"].astype(int)
    df_rent_clean = df_rent.copy()

    period_counts = df_rent_clean["data_period"].value_counts()
    print(f"\n{'=' * 50}")
    print(f"  RENTAL CLEANING SUMMARY")
    print(f"{'=' * 50}")
    print(f"  Raw rows          : {raw_rent_count:,}")
    print(f"  Final clean rows  : {len(df_rent_clean):,}")
    print(f"\n  data_period split:")
    for p in ["pre_covid", "transition", "post_covid"]:
        cnt = period_counts.get(p, 0)
        pct = cnt / len(df_rent_clean) * 100 if len(df_rent_clean) > 0 else 0
        print(f"    {p:<14}: {cnt:>5,} ({pct:>5.1f}%)")
    print(f"\n  City distribution:")
    for c, cnt in df_rent_clean["city"].value_counts().items():
        print(f"    {c:<15}: {cnt:>5,}")
    print(f"{'=' * 50}\n")

    df_rent_clean.to_csv(PROCESSED_DIR / "rent_clean.csv", index=False)
    logger.info("Rental checkpoint saved: %d rows", len(df_rent_clean))

except Exception as exc:
    logger.error("Rental processing failed: %s", exc)
    df_rent_clean = pd.DataFrame()
    rental_date_limitation = "Processing failed"

## Cell 10 -- Process RBI House Price Index

**Why:** The RBI HPI is our macro-economic signal -- a quarterly All-India
residential property price index. We compute period averages that will be
merged into the master sale dataset to give each row a macro context signal.

In [ ]:
try:
    df_hpi_raw = pd.read_csv(DATA_DIR / DATASETS["hpi"])
    logger.info("Loaded HPI: %d rows x %d cols", *df_hpi_raw.shape)

    # Rename columns
    date_col = df_hpi_raw.columns[0]
    value_col = df_hpi_raw.columns[1]
    df_hpi = df_hpi_raw.rename(columns={date_col: "date", value_col: "rbi_hpi"}).copy()
    df_hpi["date"] = pd.to_datetime(df_hpi["date"], errors="coerce")
    df_hpi = df_hpi.sort_values("date").reset_index(drop=True)

    # Rolling signals
    df_hpi["rbi_hpi_yoy"] = df_hpi["rbi_hpi"].pct_change(4) * 100
    df_hpi["rbi_hpi_qoq"] = df_hpi["rbi_hpi"].pct_change(1) * 100

    # Compute period averages
    hpi_pre = df_hpi[df_hpi["date"].dt.year <= 2019]["rbi_hpi"].mean()
    hpi_trans = df_hpi[df_hpi["date"].dt.year.between(2020, 2021)]["rbi_hpi"].mean()
    hpi_post = df_hpi[df_hpi["date"].dt.year >= 2022]["rbi_hpi"].mean()

    PERIOD_HPI_MAP.update({
        "pre_covid":  round(hpi_pre, 2),
        "transition": round(hpi_trans, 2),
        "post_covid": round(hpi_post, 2),
    })

    print(f"\n{'=' * 50}")
    print(f"  RBI HPI PERIOD AVERAGES")
    print(f"{'=' * 50}")
    print(f"  pre_covid  (<=2019) : {hpi_pre:.2f}")
    print(f"  transition (20-21)  : {hpi_trans:.2f}")
    print(f"  post_covid (>=2022) : {hpi_post:.2f}")
    print(f"  Total growth        : {((hpi_post / hpi_pre) - 1) * 100:.1f}%")
    print(f"{'=' * 50}\n")

    df_hpi_clean = df_hpi.copy()
    df_hpi_clean.to_csv(PROCESSED_DIR / "hpi_macro.csv", index=False)
    logger.info("HPI checkpoint saved: %d rows", len(df_hpi_clean))

except Exception as exc:
    logger.error("HPI processing failed: %s", exc)
    df_hpi_clean = pd.DataFrame()
    PERIOD_HPI_MAP.update({"pre_covid": 130.0, "transition": 155.0, "post_covid": 170.0})

## Cell 11 -- Build Master Sale Dataset

**Why:** All five city-level sale datasets are now individually cleaned to a
common schema. Concatenating them into one master dataframe lets us train
a single multi-city model. We also enrich each row with its RBI HPI period
average, giving the model a macro-economic signal.

In [ ]:
sale_dfs = {
    "Bengaluru": df_blr_clean,
    "Pune": df_pune_clean,
    "Delhi": df_del_clean,
    "Mumbai": df_mum_clean,
    "V21": df_v21_clean,
}

# Filter out empty dataframes
non_empty = {k: v for k, v in sale_dfs.items() if len(v) > 0}
logger.info("Merging %d non-empty sale datasets: %s", len(non_empty), list(non_empty.keys()))

df_sale_master = pd.concat(non_empty.values(), ignore_index=True)

# Add RBI HPI signal
df_sale_master["rbi_hpi_avg"] = df_sale_master["data_period"].map(PERIOD_HPI_MAP)

# Fill any unmapped periods with the overall mean
overall_hpi_mean = sum(PERIOD_HPI_MAP.values()) / len(PERIOD_HPI_MAP) if PERIOD_HPI_MAP else 150.0
df_sale_master["rbi_hpi_avg"] = df_sale_master["rbi_hpi_avg"].fillna(overall_hpi_mean)

# Print master summary
print(f"\n{'=' * 55}")
print(f"  MASTER SALE DATASET -- FINAL SUMMARY")
print(f"{'=' * 55}")
print(f"  Total rows    : {len(df_sale_master):,}")
print(f"\n  BY CITY:")
for city, cnt in df_sale_master["city"].value_counts().items():
    pct = cnt / len(df_sale_master) * 100
    print(f"    {city:<15}: {cnt:>6,} rows ({pct:>5.1f}%)")

print(f"\n  BY PERIOD:")
for period in ["pre_covid", "transition", "post_covid"]:
    cnt = (df_sale_master["data_period"] == period).sum()
    pct = cnt / len(df_sale_master) * 100
    print(f"    {period:<14}: {cnt:>6,} rows ({pct:>5.1f}%)")

print(f"\n  BY SOURCE:")
for src, cnt in df_sale_master["source"].value_counts().items():
    print(f"    {src:<20}: {cnt:>6,}")

if len(df_sale_master) > 0:
    print(f"\n  Price range   : Rs {df_sale_master['price_lakh'].min():.1f}L -- Rs {df_sale_master['price_lakh'].max():,.0f}L")
    print(f"  Avg price/sqft: Rs {df_sale_master['price_sqft'].mean():,.0f}")
    print(f"  HPI enrichment: {PERIOD_HPI_MAP}")

print(f"\n  Columns: {df_sale_master.columns.tolist()}")
print(f"{'=' * 55}\n")

# Assertions
assert len(df_sale_master) > 50000, f"Master: only {len(df_sale_master)} rows -- expected >50K"
assert "rbi_hpi_avg" in df_sale_master.columns
assert df_sale_master["rbi_hpi_avg"].isna().sum() == 0, "Unmapped HPI values remain"
assert df_sale_master["price_lakh"].isna().sum() == 0, "Null prices in master"
for col in STANDARD_COLS:
    assert col in df_sale_master.columns, f"Missing standard column: {col}"

logger.info("Master sale dataset built: %d rows, %d columns", *df_sale_master.shape)

## Cell 12 -- Create Train / Val / Drift Splits

**Why:** The `data_period` column already labels each row as pre_covid,
transition, or post_covid. We split on this label to create:
- **train_baseline** (pre_covid): model training data
- **val** (transition): validation / early-warning data
- **drift_window** (post_covid): drift detection target

A KS-test preview verifies that drift is real before Notebook 05 runs
the full drift detection pipeline.

In [ ]:
df_train = df_sale_master[df_sale_master["data_period"] == "pre_covid"].copy()
df_val = df_sale_master[df_sale_master["data_period"] == "transition"].copy()
df_drift = df_sale_master[df_sale_master["data_period"] == "post_covid"].copy()

logger.info("Split sizes -- train: %d, val: %d, drift: %d", len(df_train), len(df_val), len(df_drift))

# KS-test drift verification
train_prices = df_train["price_sqft"].dropna()
drift_prices = df_drift["price_sqft"].dropna()
ks_stat, p_value = stats.ks_2samp(train_prices, drift_prices)
shift_pct = ((drift_prices.mean() - train_prices.mean()) / train_prices.mean() * 100)

drift_confirmed = p_value < 0.05
result_text = "REJECT H0 -- DRIFT CONFIRMED" if drift_confirmed else "FAIL TO REJECT -- WEAK DRIFT"

print(f"\n{'=' * 50}")
print(f"  DRIFT VERIFICATION -- KS TEST PREVIEW")
print(f"{'=' * 50}")
print(f"  Train mean price/sqft : Rs {train_prices.mean():,.0f}")
print(f"  Drift mean price/sqft : Rs {drift_prices.mean():,.0f}")
print(f"  Price shift           : {'+' if shift_pct > 0 else ''}{shift_pct:.1f}%")
print(f"  KS Statistic          : {ks_stat:.4f}")
print(f"  p-value               : {p_value:.6f}")
print(f"")
print(f"  H0: distributions are identical")
print(f"  Result: {result_text}")
print(f"{'=' * 50}\n")

# Assertions
assert drift_prices.mean() > train_prices.mean(), \
    "Drift prices should be higher than train prices (post-COVID appreciation)"
assert len(df_train) > len(df_val), "Train set should be larger than validation set"
assert len(df_drift) > 1000, f"Drift window too small: {len(df_drift)}"
assert len(df_train) > 1000, f"Train set too small: {len(df_train)}"

logger.info("Drift verification: KS=%.4f, p=%.6f, shift=%.1f%% -- %s",
            ks_stat, p_value, shift_pct, result_text)

## Cell 13 -- Save All Files & Preprocessing Report

**Why:** Persisting every cleaned dataset, split, and the preprocessing report
ensures downstream notebooks can pick up exactly where this notebook left off
without re-running the entire cleaning pipeline.

In [ ]:
# -- Save sale splits -------------------------------------------------------
df_sale_master.to_csv(PROCESSED_DIR / "master_clean.csv", index=False)
df_train.to_csv(PROCESSED_DIR / "train_baseline.csv", index=False)
df_val.to_csv(PROCESSED_DIR / "val.csv", index=False)
df_drift.to_csv(PROCESSED_DIR / "drift_window.csv", index=False)

# -- Save rental splits ----------------------------------------------------
if len(df_rent_clean) > 0:
    df_rent_train = df_rent_clean[df_rent_clean["data_period"] == "pre_covid"].copy()
    df_rent_drift = df_rent_clean[df_rent_clean["data_period"] == "post_covid"].copy()
    df_rent_train.to_csv(PROCESSED_DIR / "rent_train.csv", index=False)
    df_rent_drift.to_csv(PROCESSED_DIR / "rent_drift.csv", index=False)
else:
    df_rent_train = pd.DataFrame()
    df_rent_drift = pd.DataFrame()

# -- Preprocessing report --------------------------------------------------
preprocessing_report = {
    "generated_at": datetime.now().isoformat(),
    "project": "PropertyIQ",
    "notebook": "02_preprocessing",
    "total_sale_rows": int(len(df_sale_master)),
    "cities_covered": sorted(df_sale_master["city"].unique().tolist()),
    "train_rows": int(len(df_train)),
    "val_rows": int(len(df_val)),
    "drift_rows": int(len(df_drift)),
    "rent_clean_rows": int(len(df_rent_clean)),
    "rent_train_rows": int(len(df_rent_train)),
    "rent_drift_rows": int(len(df_rent_drift)),
    "hpi_rows": int(len(df_hpi_clean)) if "df_hpi_clean" in dir() else 0,
    "ks_statistic": round(float(ks_stat), 6),
    "p_value": round(float(p_value), 8),
    "drift_confirmed": drift_confirmed,
    "train_mean_price_sqft": round(float(train_prices.mean()), 2),
    "drift_mean_price_sqft": round(float(drift_prices.mean()), 2),
    "price_shift_pct": round(float(shift_pct), 2),
    "hpi_period_map": PERIOD_HPI_MAP,
    "mumbai_lac_rows": mumbai_lac_rows,
    "mumbai_cr_rows": mumbai_cr_rows,
    "rental_date_limitation": rental_date_limitation if "rental_date_limitation" in dir() else "N/A",
    "sanity_bounds": {
        "price_min_lakh": PRICE_MIN_LAKH,
        "price_max_lakh": PRICE_MAX_LAKH,
        "sqft_min": SQFT_MIN,
        "sqft_max": SQFT_MAX,
    },
}

report_path = OUTPUT_DIR / "preprocessing_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(preprocessing_report, f, indent=2, ensure_ascii=False, default=str)

assert report_path.exists(), f"Preprocessing report not saved to {report_path}"

# -- Final manifest ---------------------------------------------------------
files_saved = {
    "master_clean.csv": len(df_sale_master),
    "train_baseline.csv": len(df_train),
    "val.csv": len(df_val),
    "drift_window.csv": len(df_drift),
    "rent_clean.csv": len(df_rent_clean),
    "rent_train.csv": len(df_rent_train),
    "rent_drift.csv": len(df_rent_drift),
    "hpi_macro.csv": len(df_hpi_clean) if "df_hpi_clean" in dir() else 0,
}

print(f"\n{'=' * 50}")
print(f"  ALL FILES SAVED -- data/processed/")
print(f"{'=' * 50}")
for fname, rows in files_saved.items():
    marker = "[OK]" if rows > 0 else "[EMPTY]"
    print(f"  {marker} {fname:<25} {rows:>6,} rows")
print(f"\n  outputs/preprocessing_report.json saved [OK]")
print(f"{'=' * 50}\n")

logger.info("Notebook 02 complete -- all files saved to %s", PROCESSED_DIR)